In [ ]:
# HƯỚNG DẪN SỬ DỤNG:

# BƯỚC 1: Trên drive: Chọn file audio -> Ấn vào Share -> Copy link
# BƯỚC 2: Paste link vào ô thứ 2 -> Copy đoạn giữa /d/ và /view?
# BƯỚC 3: Paste id vào ô thứ 3, vào vị trí giữa --id và -O

# BƯỚC 4: Runtime -> Run All (Thời gian chạy -> chạy tất cả)
# BƯỚC 5: Files -> Tải transcript.txt -> Disconnect and delete Runtime (Ngắt kết nối và xóa thời gian chạy) trước khi chạy lần sau

In [ ]:
# https://drive.google.com/file/d/1aYVZAAc4cssdYywmrvl5PG16qNNXzw_x/view?usp=sharing

In [ ]:
!gdown --id 1aYVZAAc4cssdYywmrvl5PG16qNNXzw_x -O audio.m4a

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1aYVZAAc4cssdYywmrvl5PG16qNNXzw_x
From (redirected): https://drive.google.com/uc?id=1aYVZAAc4cssdYywmrvl5PG16qNNXzw_x&confirm=t&uuid=9b479f31-66b6-41bf-b519-1a8a4583eb61
To: /content/audio.m4a
100% 165M/165M [00:04<00:00, 33.8MB/s]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pydub import AudioSegment

input_m4a_file = '/content/audio.m4a'
output_wav_file = '/content/audio.wav'

try:
    audio = AudioSegment.from_file(input_m4a_file, format="m4a")

    # --- SỬA LỖI TẠI ĐÂY: set_channels(1) ---
    audio = audio.set_channels(1)

    # Export it as a wav file
    audio.export(output_wav_file, format="wav")
    print(f"Conversion successful! Mono WAV file saved to: {output_wav_file}")
except Exception as e:
    print(f"An error occurred during conversion: {e}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Conversion successful! Mono WAV file saved to: /content/audio.wav


In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# ==========================================
# CẤU HÌNH (CONFIGURATION)
# ==========================================
INPUT_AUDIO_PATH = "audio.wav"
OUTPUT_TRANSCRIPT_PATH = "transcript.txt"
SEGMENT_DURATION_MS = 120 * 1000

# ==========================================
# BƯỚC 1: LOAD MÔ HÌNH (SETUP MODEL)
# ==========================================
print("--- Bước 1: Đang tải mô hình Whisper... ---")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_name_or_path = "bofenghuang/whisper-large-v3-french"

processor = AutoProcessor.from_pretrained(model_name_or_path)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_name_or_path,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

# Init pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=30,
    max_new_tokens=128,
    generate_kwargs={"language": "french"} # Gợi ý model rằng đây là tiếng Pháp
)

print(f"Mô hình đã tải xong trên thiết bị: {device}")

# ==========================================
# BƯỚC 2 & 3: CẮT FILE VÀ TRANSCRIBE
# ==========================================

print(f"--- Đang tải file âm thanh: {INPUT_AUDIO_PATH} ---")
# Load toàn bộ file audio bằng pydub
audio = AudioSegment.from_file(INPUT_AUDIO_PATH)
audio_length_ms = len(audio)

print(f"Tổng thời lượng: {audio_length_ms / 1000} giây")
print("Bắt đầu xử lý từng đoạn âm thanh...")

transcript_lines = []
file_number = 1

for i in range(0, audio_length_ms, SEGMENT_DURATION_MS):
    # Xác định thời gian bắt đầu và kết thúc của đoạn
    start_time_ms = i
    end_time_ms = min(i + SEGMENT_DURATION_MS, audio_length_ms)

    # Cắt đoạn âm thanh
    segment = audio[start_time_ms:end_time_ms]

    # Định dạng thời gian cho việc ghi log/file txt (Phút:Giây)
    start_minutes = start_time_ms // (60 * 1000)
    start_seconds = (start_time_ms % (60 * 1000)) // 1000

    end_minutes = end_time_ms // (60 * 1000)
    end_seconds = (end_time_ms % (60 * 1000)) // 1000

    time_range_str = f"{start_minutes:02}m{start_seconds:02}s - {end_minutes:02}m{end_seconds:02}s"

    print(f"  > Đang xử lý đoạn {file_number} ({time_range_str})...")

    # Lưu đoạn tạm thời để pipeline có thể đọc
    temp_segment_path = f"temp_segment_{file_number}.wav"
    segment.export(temp_segment_path, format="wav")

    # Transcribe đoạn âm thanh
    try:
        # Gọi pipeline đã khởi tạo ở Bước 1
        result = pipe(temp_segment_path)
        transcribed_text = result["text"].strip()
    except Exception as e:
        transcribed_text = f"[LỖI XỬ LÝ ĐOẠN NÀY: {str(e)}]"
        print(f"    Lỗi: {e}")

    # Xóa file tạm để giải phóng ổ cứng
    if os.path.exists(temp_segment_path):
        os.remove(temp_segment_path)

    # ==========================================
    # BƯỚC 4: ĐỊNH DẠNG NỘI DUNG
    # ==========================================
    header = f"Đoạn {file_number} ({time_range_str}):"
    transcript_lines.append(header)
    transcript_lines.append(transcribed_text)
    transcript_lines.append("-" * 20) # Dòng kẻ ngang phân cách

    print(f"    -> Xong đoạn {file_number}")
    file_number += 1

    # Dọn RAM GPU sau mỗi lần chạy
    torch.cuda.empty_cache()

# ==========================================
# BƯỚC 5: LƯU FILE KẾT QUẢ
# ==========================================
print(f"--- Bước 5: Đang lưu toàn bộ bản transcript vào file: {OUTPUT_TRANSCRIPT_PATH} ---")

with open(OUTPUT_TRANSCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(transcript_lines))

print("Quá trình hoàn tất!")

--- Bước 1: Đang tải mô hình Whisper... ---


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

    -> Xong đoạn 35
  > Đang xử lý đoạn 36 (70m00s - 72m00s)...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

    -> Xong đoạn 36
  > Đang xử lý đoạn 37 (72m00s - 73m11s)...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

    -> Xong đoạn 37
--- Bước 5: Đang lưu toàn bộ bản transcript vào file: transcript.txt ---
Quá trình hoàn tất!
